####1. Requirement
Read data from students_offline.csv file and load into offline_students_raw table.

In [0]:
offline_students_schema = "ID string, FirstName string, LastName string, Address string, Skills string, Contacts string"

offline_students_raw_df = (
    spark.read.format("csv")
        .option("header", "true")
        .option("quote", "\"")
        .option("escape", "\"")
        .schema(offline_students_schema)
        .load("/Volumes/dev_catalog/spark_db/datasets/spark_programming/data/students_offline.csv")
)

# Rename columns to match Delta table schema
offline_students_raw_df = offline_students_raw_df
    .withColumnRenamed("ID", "id") \
    .withColumnRenamed("FirstName", "first_name") \
    .withColumnRenamed("LastName", "last_name") \
    .withColumnRenamed("Address", "address") \
    .withColumnRenamed("Skills", "skills") \
    .withColumnRenamed("Contacts", "contacts")

# display(offline_students_raw_df)
offline_students_raw_df.write.mode("overwrite").saveAsTable("dev_catalog.spark_db.offline_students_raw")

In [0]:
%sql
use catalog `dev_catalog`; 
select * from `spark_db`.`offline_students_raw` limit 100;

####2. Analysis Requirement
We want to know country wise student count.

In [0]:
%sql

with offline_students(
  select id, from_json(address,
      """struct<AddressLine1 string,
        AddressLine2 string,
        City string,
        Country string,
        Pin string,
        State string>
      """) as address
  from dev_catalog.spark_db.offline_students_raw
)
select address.country, count(*) as count
from offline_students
group by address.country

####3. Requirement
Prepare an offline_students table which is ready for analysis

Complex Data Types in Spark
1. Struct
2. Array
3. Map

In [0]:
from pyspark.sql.functions import from_json

address_schema = "struct<AddressLine1 string, AddressLine2 string, City string, Country string, Pin string, State string>"
skills_schema = "array<struct<Skill string, YearsOfExperience string>>"
contacts_schema = "map<string, string>"

offline_students_df = (
    offline_students_raw_df.withColumns({
        "address": from_json("address", address_schema),
        "skills": from_json("skills", skills_schema),
        "contacts": from_json("contacts", contacts_schema)
    })
)

#offline_students_df.display()
offline_students_df.write.mode("overwrite").saveAsTable("dev_catalog.spark_db.offline_students")

####4. Requirement
Perform the following analysis
1. What is country wise student count.
2. Find all students with more than 1 years of Spark knowledge
3. Find all students who didn't provide phone or whatsapp

4.1 What is country wise student count.

In [0]:
%sql

select address.Country, count(*) as count
from dev_catalog.spark_db.offline_students
group by address.Country

4.2 Find all students with more than 1 years of Spark knowledge

In [0]:
%sql

with offline_students_skills(
   select id, first_name, last_name, explode(skills) as skills
   from dev_catalog.spark_db.offline_students
)
select id, first_name, last_name, skills.*
from offline_students_skills
where skills.Skill like "%Spark%" and skills.YearsOfExperience > 1

4.3 Find all students who didn't provide phone or whatsapp

In [0]:
%sql

select id, first_name, last_name, contacts['email']
from dev_catalog.spark_db.offline_students
where contacts['phone'] is null and contacts['whatsapp'] is null